# `sch_straighten`：原理图「横平竖直」后处理 —— 入库评估 / 库化 / 流水线

这个 notebook 回答三个问题：

1. **能入库吗？** —— 能。它是 `tools/sch_straighten.py`，判据与既有唯一实现同源
   （`sch_box.pins_of` 量脚位 ✓），**默认只检查、加 `--apply` 才写文件** ✓。
2. **能当原理图的「后处理步骤」吗？** —— 能，而且是**设计目的**：
   `读图 → （改/布线）→ sch_straighten → 再核 → 出图` ✓（见 §7）。
3. **真能把图弄直吗？** —— 本 notebook **自己量** ✓，不信工具自己的话 ✓
   （规矩：「不许自证」✗ —— 工具的报账只是线索，结论要用独立实现 + 人的眼睛复核 ✓）。

**真实病例**（2026-10-03，`u270b_routed.fzz`）：一小段竖直干线由 4 段拼成，
其中 2 段各斜 **0.6721 sketch 单位 = 0.19 mm** ✗ —— 整图看不出来、一放大就露 ✗。
本 notebook 用**改前 / 改后**两份真实文件，独立量「正交度」并对比。

> 依赖：numpy / pandas / pillow / cairosvg 本机已有；
> matplotlib / networkx 若缺失会自动降级（PIL 画图 + 内建图结构），notebook 仍可跑完 ✓。


## 1. 导入依赖 + 找到工具与样例

- 工具 = 同目录的 `sch_straighten.py` ✓（本 notebook 用 `subprocess` 调它的 CLI ✓ —— 
  它**还没有 `main()` 守卫**，直接 `import` 会执行命令行 ✗，§6 会专门讲这件事 ✓）。
- 样例 = 项目仓里那份真实草图；找不到就退化成内建的小样例 ✓（换机器也能跑完 ✓）。


In [ ]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

# ── 可选依赖：有就用、没有就降级 ✓（本机没装 matplotlib / networkx ✓）──
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False
try:
    import networkx as nx
    HAVE_NX = True
except Exception:
    HAVE_NX = False
from PIL import Image, ImageDraw

HERE = Path.cwd() if (Path.cwd() / "sch_straighten.py").exists() else Path(
    r"f:\git\fritzing-parts-langhua\tools")
TOOL = HERE / "sch_straighten.py"
assert TOOL.exists(), f"找不到工具：{TOOL}"

# 真实样例（改前 / 改后）：项目仓里那份草图的备份与现值 ✓
W = Path(r"f:\git\AuroraTessellation-NFC\hardware\pixel\_work")
SKETCH_AFTER = W / "u270b_routed.fzz"                 # 已经拉直（人工修过 ✓）
SKETCH_BEFORE = W / "u270b_routed.fzz.prestraight"    # 拉直**之前**的备份 ✓
CANDIDATES = [W / n for n in ("u270b_routed.fzz.prestraight", "u270b_routed.fzz.bak-0402",
                              "u270b_routed.fzz.presch", "u270b_routed.fzz")]

TMP = Path(tempfile.mkdtemp(prefix="sch_straighten_nb_"))
SK_U_PER_MM = 25.4 / 90.0      # 1 sketch 单位 = 1/90 in ✓
TOL_MM = 0.25                  # "本该对齐"的容差 ✓（与工具默认一致 ✓）
TOL = TOL_MM * SK_U_PER_MM
KI = 10.0                      # 另一轴 ≥ KI× 这一轴 ⇒ 才算"本该横平竖直" ✓

print("工具      :", TOOL)
print("matplotlib:", HAVE_MPL, "｜networkx:", HAVE_NX, "｜numpy:", np.__version__)
print("改前样例  :", SKETCH_BEFORE.exists(), SKETCH_BEFORE)
print("改后样例  :", SKETCH_AFTER.exists(), SKETCH_AFTER)
print("临时目录  :", TMP)


## 2. 把草图读成「图」：节点 = 端点，边 = 导线

解析器**自己写一份** ✗ —— 故意的 ✓：这样后面量出来的数字**不依赖工具的内部实现** ✓
（工具自己也有一份解析 ✓，两份互为对照 ✓，规号「不许自证」✗）。

`<fz>` 里一根线长这样（实测 ✓）：

```xml
<schematicView layer="schematicTrace">
  <geometry z="…" x="162" y="135" x1="0" y1="0" x2="-0.6721" y2="22.8007" wireFlags="128"/>
```

两个端点 = `(x+x1, y+y1)` 与 `(x+x2, y+y2)` ✓；`wireFlags & 128` 才算**原理图的铜** ✓
（`& 4` 是 PCB 铜、`& 64` 是面包板 —— 不懂 `wireFlags` 就会把 PCB 走线当成原理图线 ✗）。


In [ ]:
VIEW = "schematicView"
BIT = 128          # 原理图的铜位 ✓（面包板 64 / PCB 4 ✓）


def fnum(attrs, key, default=0.0):
    """取数值属性 ✓ —— ★ 只能走 `.get()` ✓：
    `ElementTree` 的 `Element[...]` 要的是**子元素下标**（整数 ✗），`elem["x"]` 抛
    `TypeError: element indices must be integers` ✗ ⇒ 被 `except` 吞掉 ⇒ **全读出 0** ✗✗
    （实测踩过：正交度全算成 `nan` ✗ = 这一个坑 ✓）。dict / Element 都用 `.get` ✓ 通吃 ✓。
    """
    try:
        v = attrs.get(key)
        return float(v) if v is not None else default
    except (AttributeError, TypeError, ValueError):
        return default


def fz_text(path):
    """`.fzz`(zip) 与 `.fz`(纯 xml) 都收 ✓ —— ★ 用**魔数**认 zip ✗ 别看扩展名 ✗"""
    p = Path(path)
    if p.read_bytes()[:2] == b"PK":
        z = zipfile.ZipFile(p)
        name = [n for n in z.namelist() if n.endswith(".fz")][0]
        return z.read(name).decode("utf-8")
    return p.read_text(encoding="utf-8")


def read_wires(path):
    """→ DataFrame[title, ax, ay, bx, by] ✓（独立实现 ✓，只认原理图铜 ✓）"""
    root = ET.fromstring(fz_text(path))
    rows = []
    for inst in root.iter("instance"):
        if "Wire" not in (inst.get("moduleIdRef") or ""):
            continue
        views = inst.find("views")
        sv = views.find(VIEW) if views is not None else None
        g = sv.find("geometry") if sv is not None else None
        if g is None:
            continue
        fl = g.get("wireFlags")
        if fl is not None and not (int(fl) & BIT):
            continue                       # 这条在本视图不算铜 ⇒ 跳过 ✓
        x, y = fnum(g, "x"), fnum(g, "y")
        rows.append(dict(title=(inst.findtext("title") or "").strip(),
                         ax=x + fnum(g, "x1"), ay=y + fnum(g, "y1"),
                         bx=x + fnum(g, "x2"), by=y + fnum(g, "y2")))
    return pd.DataFrame(rows)


def build_graph(df):
    """原理图 ⇒ 图（节点 = 端点 ✓，边 = 导线 ✓）；有 networkx 就用它 ✓，否则退化 dict ✓"""
    def key(x, y):
        return (round(x, 4), round(y, 4))
    if HAVE_NX:
        g = nx.Graph()
        for r in df.itertuples():
            g.add_edge(key(r.ax, r.ay), key(r.bx, r.by), wire=r.title)
        return g, "networkx.Graph"
    g = {}
    for r in df.itertuples():
        g.setdefault(key(r.ax, r.ay), []).append((key(r.bx, r.by), r.title))
        g.setdefault(key(r.bx, r.by), []).append((key(r.ax, r.ay), r.title))
    return g, "内建 dict 邻接表（没装 networkx ✓）"


if SKETCH_BEFORE.exists():
    DF_BEFORE = read_wires(SKETCH_BEFORE)
    SRC_LABEL = SKETCH_BEFORE.name
else:                                       # 换机器 ⇒ 用内建小样例 ✓
    DF_BEFORE = pd.DataFrame([
        dict(title="w1", ax=0.0, ay=0.0, bx=0.0, by=60.0),          # 竖 ✓
        dict(title="w2", ax=0.0, ay=60.0, bx=0.6721, by=100.0),     # 想竖、却斜了 0.19mm ✗
        dict(title="w3", ax=0.6721, ay=100.0, bx=60.0, by=100.0),   # 横 ✓
    ])
    SRC_LABEL = "内建小样例（未找到真实草图）"

G, GNAME = build_graph(DF_BEFORE)
N_NODES = len(G.nodes) if HAVE_NX else len(G)
print(f"读入 {SRC_LABEL}：导线 {len(DF_BEFORE)} 根 ✓｜图结构 = {GNAME} ✓｜图节点 {N_NODES} 个 ✓")
DF_BEFORE.head()


## 3. 跑工具：先「检查」，再「写到别处」

工具的口径 ✓：

- **默认 = 只报告，不写文件** ✓（安全 ✓）；
- `--apply` = 原地改 + 备份 `.bak-straighten` ✓；
- `--out <新文件>` = 写到别处、原文件不动 ✓ ← **后处理流水线应该用这个** ✓。

四条规则（详见工具文件头 ✓）：R1 端点吸脚 ✓ / R4 端点归并 ✓ / R2 近轴归正 ✓ /
R3 零件归列 ✓；四条安全阀 —— 脚位未知 ✗、零件多脚不一致 ✗、线端是结点 ✗、两端份量相同 ✗ ⇒ 一律**不动、只报** ✓。


In [ ]:
def run_tool(src, out=None, extra=()):
    """调工具的 CLI ✓ → (returncode, stdout) ✓"""
    cmd = [sys.executable, str(TOOL), str(src), *extra]
    if out is not None:
        cmd += ["--out", str(out)]
    p = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    return p.returncode, (p.stdout or "") + (p.stderr or "")


def check(src):
    """只检查 ✓（不写文件 ✓）"""
    return run_tool(src)


def straighten_to(src, out):
    """把 `src` 拉直后写到 `out` ✓（原文件不动 ✓）"""
    return run_tool(src, out=out)


SRC = SKETCH_BEFORE if SKETCH_BEFORE.exists() else None
if SRC is not None:
    rc, txt = check(SRC)
    print(f"【只检查】rc={rc} ✓")
    print("\n".join(txt.splitlines()[:14]))
else:
    print("（没有真实草图 ⇒ 跳过：§4/§5 仍用内建小样例演示 ✓）")


## 4. 看图：改前 / 改后并排

- 灰线 = 导线 ✓；**红线 = 「本该横平竖直、却没对齐」** ✗（判据见 §5 ✓）。
- 有 matplotlib 就用它（带坐标轴 ✓）；没有就用 PIL 画一张 PNG 出来看 ✓。


In [ ]:
def ortho_flags(df):
    """逐根判：① 是不是「本该横平竖直」？② 对齐了没？→ 两个布尔列 + 偏差列 ✓"""
    meant, aligned, dev = [], [], []
    for r in df.itertuples():
        dx, dy = abs(r.bx - r.ax), abs(r.by - r.ay)
        small, big = min(dx, dy), max(dx, dy)
        m = (big > 0) and (big >= KI * max(small, 1e-9))     # 另一轴 ≥ KI 倍 ⇒ 该是横/竖 ✓
        meant.append(bool(m))
        aligned.append(bool(m and small <= TOL))
        dev.append(small if m else np.nan)
    out = df.copy()
    out["meant"], out["aligned"], out["dev"] = meant, aligned, dev
    return out


def draw(df, out_png, title):
    """画一张图：灰 = 普通线 ✓，红 = 「该直没直」 ✗（有 matplotlib 就用 ✓）"""
    d = ortho_flags(df)
    if HAVE_MPL:
        fig, ax = plt.subplots(figsize=(6, 6))
        for r in d.itertuples():
            bad = r.meant and not r.aligned
            ax.plot([r.ax, r.bx], [-r.ay, -r.by], color=("red" if bad else "#999999"),
                    lw=(2.0 if bad else 1.0), solid_capstyle="round")
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=10)
        ax.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(out_png, dpi=110)
        plt.close(fig)
    else:                                                     # PIL 降级 ✓
        xs = np.concatenate([d.ax.values, d.bx.values])
        ys = np.concatenate([d.ay.values, d.by.values])
        w = float(np.ptp(xs)) or 1.0
        h = float(np.ptp(ys)) or 1.0
        S = 900.0
        img = Image.new("RGB", (int(S), int(S * h / w + 40)), "white")
        dr = ImageDraw.Draw(img)
        sc = S / w

        def P(x, y):
            return ((x - xs.min()) * sc, (y - ys.min()) * sc + 20.0)

        for r in d.itertuples():
            bad = r.meant and not r.aligned
            dr.line([P(r.ax, r.ay), P(r.bx, r.by)],
                    fill=("red" if bad else (150, 150, 150)), width=(4 if bad else 2))
        dr.text((8, 4), title, fill="black")
        img.save(out_png)
    return out_png


OUT = TMP / "straightened.fzz"
if SRC is not None:
    rc, txt = straighten_to(SRC, OUT)
    DF_AFTER = read_wires(OUT)
    print(f"【--out 写一份】rc={rc} ✓ → {OUT.name}（原文件未动 ✓）")
    print(f"改前 {len(DF_BEFORE)} 根导线 ✓｜改后 {len(DF_AFTER)} 根 ✓"
          f"（根数应当不变 ✓：{len(DF_BEFORE) == len(DF_AFTER)} ✓）")
else:
    _d = ortho_flags(DF_BEFORE).copy()
    _d.loc[_d["meant"], "bx"] = _d.loc[_d["meant"], "ax"].values      # 手工造一个「改后」版本 ✓
    DF_AFTER = _d
    print("（内建小样例：手工造一个「改后」版本 ✓）")

P1 = draw(DF_BEFORE, TMP / "before.png", "before")
P2 = draw(DF_AFTER, TMP / "after.png", "after")
try:                                        # notebook 里能内嵌显示 ✓
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=str(P1)))
    display(IPyImage(filename=str(P2)))
except Exception as _e:                     # 当脚本跑时只是打印路径 ✓
    print("（无 IPython 显示 ⇒ 图在：）", P1.name, P2.name, "｜", _e)


## 5. 量一量：正交度（自己算 ✓，不信工具自己的报账 ✓）

- **分母** = 「本该横平竖直」的导线数（另一轴 ≥ 10× 这一轴 ✓ —— 真的斜线不算 ✗，那是设计意图 ✓）；
- **分子** = 其中**已对齐**的（小的那一轴 ≤ 0.25 mm ✓）；
- **正交度** = 分子 / 分母 ✓；另报**最大偏差**（mm ✓ —— 人眼能看见的就是它 ✓）。

★ 口径提醒：Fritzing 的连接是**显式**记在 `<connects>` 里的 ✓ ⇒ 本项量的是**图好不好看/好不好读** ✓，
**不是**电气连通性 ✗（连通性另有 `render_sch.py` 的 (A)/(B) 判据 ✓）。


In [ ]:
def ortho_score(df):
    """→ dict(meant, aligned, score, worst_mm, offenders[list]) ✓（独立实现 ✓）"""
    d = ortho_flags(df)
    m = d[d["meant"]]
    off = m[~m["aligned"]].sort_values("dev", ascending=False)
    return dict(n_wires=len(d), meant=len(m), aligned=int(m["aligned"].sum()),
                score=(float(m["aligned"].mean()) if len(m) else float("nan")),
                worst_mm=(float(off["dev"].iloc[0]) * SK_U_PER_MM if len(off) else 0.0),
                offenders=[(r.title, round(r.dev * SK_U_PER_MM, 4)) for r in off.itertuples()])


sb, sa = ortho_score(DF_BEFORE), ortho_score(DF_AFTER)
rows = []
for name, s in (("改前（真实备份 .prestraight）", sb), ("改后（工具 --out 出来的）", sa)):
    rows.append(dict(样本=name, 导线=s["n_wires"], 该横平竖直=s["meant"],
                     已对齐=s["aligned"], 正交度=round(s["score"], 4),
                     最大偏差_mm=round(s["worst_mm"], 4), 还歪的根数=len(s["offenders"])))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
if sb["offenders"]:
    print("\n改前还歪着的（title, 偏差 mm）：", sb["offenders"][:8])
if sa["offenders"]:
    print("改后仍歪着的（工具**故意不动**的 ✓）：", sa["offenders"][:8])
print(f"\n★ 正交度 {sb['score']:.3f} → {sa['score']:.3f} ✓"
      f"（+{sa['score'] - sb['score']:.3f} ✓）")


## 6. 「入库」评估：结论 —— **已入库 ✓**，而且已经**库化** ✓

| 项 | 状态 |
|---|---|
| 层位 | ✅ `tools/sch_straighten.py` ✓（与 `render_sch.py` / `sch_style_check.py` 同层 ✓），`tools/README.md` 已登记 ✓、`AGENTS.md` §11 有指针 ✓ |
| 脚本入口 | ✅ `main(argv=None)` + `if __name__ == "__main__": sys.exit(main())` 守卫 ⇒ `import` **不再**执行命令行 ✓ |
| 库 API | ✅ `straighten_file(src, out=None, tol_mm=0.25, view="schematicView", apply=False, quiet=True) -> dict` ✓ |
| 返回值 | ✅ `rc`（0 / 2）· `changes` · `plan` · `skipped` · `v1_before` · `v1_after` · `still_slanted` · `wrote` · `out` · `text` ✓ |
| 判据实现 | ✅ 复用 `sch_box.pins_of` / `A_of` ✓ —— 与渲染器**同一把尺子** ✓（不是两套 ✗） |
| 安全性 | ✅ 默认只检查 ✓；`<connects>` 条数变了 ✗ / 改出的 XML 解析不了 ✗ ⇒ **不写文件** ✓ |

★ 库化时**行为必须逐字不变** ✗ ⇒ 做法：拿库化**前**跑出来的 `.fz` 当基准 ✓、库化后再跑一遍
**逐字节比对** ✓（实测**相同** ✓）。这比"我觉得没改坏"硬 ✓。


In [ ]:
RE_N_CHANGE = re.compile(r"改动 (\d+) 处")
RE_V1 = re.compile(r"V1 线端不在它声明的脚上 \*\*(\d+) 处\*\*")
RE_V3 = re.compile(r"V3 还「该直没直」 \*\*(\d+) 处\*\*")


def straighten_parsed(src, out=None):
    """CLI 跑一遍、把关键数字解析出来 ✓（§3/§7 用它演示"命令行怎么用" ✓）"""
    rc, txt = run_tool(src, out=out)
    g = lambda rx, d=-1: (int(rx.search(txt).group(1)) if rx.search(txt) else d)   # noqa: E731
    return dict(rc=rc, changes=g(RE_N_CHANGE), v1=g(RE_V1), v3=g(RE_V3),
                wrote=("写入" in txt), text=txt)


# ── 直接用**库 API** ✓（比解析 stdout 干净 ✓；两者结果应当一致 ✓）──
sys.path.insert(0, str(HERE))
import sch_straighten as SS                                          # noqa: E402  ★ import 无副作用 ✓

if SRC is not None:
    r_cli = straighten_parsed(SRC, TMP / "cli_demo.fzz")
    r_api = SS.straighten_file(SRC, out=TMP / "api_demo.fzz", quiet=True)
    print("命令行：", {k: r_cli[k] for k in ("rc", "changes", "v1", "v3", "wrote")})
    print("库 API：", {k: r_api[k] for k in ("rc", "changes", "v1_before", "v1_after", "wrote")})
    print("  plan[0]  ：", r_api["plan"][0] if r_api["plan"] else "（无 ✓）")
    print("  skipped  ：", r_api["skipped"][0] if r_api["skipped"] else "（无 ✓）")
    print("  还歪的   ：", r_api["still_slanted"], "← 安全阀**明说**拦下的那根 ✓")
    print("★ 两者一致 ✓：", (r_cli["changes"] == r_api["changes"] and r_cli["v1"] == r_api["v1_after"]))
else:
    print("（没有真实草图 ⇒ 跳过 API 演示 ✓）")


## 7. 当「后处理步骤」：一条流水线

```
草图 ──▶ ① 布线/改图（本项目另有生成器）  ──▶ ② sch_straighten（后处理）  ──▶ ③ 独立复核  ──▶ 出图/交付
```

要点 ✓：

- 后处理**只动几何**（端点坐标 / 零件位置）✓，**一个字都不碰 `<connects>`** ✓
  —— 电气连接是 Fritzing 的「权威」✗，几何只是**好看** ✓（§11 已认证 ✓）。
- 后处理**必须默认只检查** ✓、写文件要留备份 ✓（工具已经这么做 ✓）。
- 后处理之后**必须复核** ✓：① 线端还在不在它声明的脚上（V1 不许变多 ✗）；
  ② 还剩下几处"该直没直"（V3，写清 mm ✓）；③ 声明表条数没变 ✓。


In [ ]:
def pipeline(src, out_dir, verbose=True):
    """① 复制 → ② 后处理（工具 --out ✓）→ ③ 独立复核 ✓（自己量 ✓）"""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stage1 = out_dir / (Path(src).stem + ".stage1.fzz")
    shutil.copy2(src, stage1)                     # ① 假装这里刚布完线 ✓
    stage2 = out_dir / (Path(src).stem + ".stage2.fzz")
    r = straighten_parsed(stage1, stage2)         # ② 后处理 ✓
    if not r["wrote"]:
        stage2 = stage1                           # 没改动 ⇒ 直接进下一步 ✓
    before, after = ortho_score(read_wires(stage1)), ortho_score(read_wires(stage2))
    if verbose:
        print(f"  ① 输入 {Path(src).name}：该横平竖直 {before['meant']} 根 ✓，"
              f"正交度 {before['score']:.3f} ✓，最歪 {before['worst_mm']:.4f} mm ✗")
        print(f"  ② 后处理：改动 {r['changes']} 处 ✓，V1 {r['v1']} ✓，V3 {r['v3']} ✓")
        print(f"  ③ 复核：正交度 {after['score']:.3f} ✓（+{after['score'] - before['score']:.3f} ✓），"
              f"最歪 {after['worst_mm']:.4f} mm ✓")
    return dict(src=str(src), stage1=stage1, stage2=stage2, before=before, after=after, tool=r)


if SRC is not None:
    print("跑一遍流水线（输入 = 真实草图「改前」版 ✓）：")
    res = pipeline(SRC, TMP / "pipe")
else:
    print("（没有真实草图 ⇒ 流水线跳过 ✓）")


## 8. 批量跑：多张草图 + 「独立复核」

样本 = 项目仓里能找到的**每一份真实草图/备份** ✓（换机器为空时退化成内建小样例 ✓）。
每一份都走 §7 的流水线，并把**前后正交度**并排列出来 ✓。

★ 这一格同时是**对工具的不信任投票** ✓：正交度由本 notebook 自己算 ✓；
工具的 `V1` 自查只作参考 ✓ —— 若两者打架，**以人的眼睛（§4 的图）为准** ✓。


In [ ]:
cases = [p for p in CANDIDATES if p.exists()]
if not cases:
    cases = [None]                                   # 内建小样例 ✓（不跑工具 ✓）
print("样本 %d 份 ✓：" % len(cases))
rows = []
for c in cases:
    if c is None:
        b = ortho_score(DF_BEFORE)
        a = ortho_score(DF_AFTER)
        rows.append(dict(样本="内建小样例", 改动="（未跑工具）", 正交度前=round(b["score"], 4),
                         正交度后=round(a["score"], 4), 提升=round(a["score"] - b["score"], 4),
                         最歪前_mm=round(b["worst_mm"], 4), 最歪后_mm=round(a["worst_mm"], 4),
                         工具V1="—", 工具V3="—"))
        continue
    res = pipeline(c, TMP / "batch" / c.name, verbose=False)
    b, a = res["before"], res["after"]
    rows.append(dict(样本=c.name, 改动=res["tool"]["changes"],
                     正交度前=round(b["score"], 4), 正交度后=round(a["score"], 4),
                     提升=round(a["score"] - b["score"], 4),
                     最歪前_mm=round(b["worst_mm"], 4), 最歪后_mm=round(a["worst_mm"], 4),
                     工具V1=res["tool"]["v1"], 工具V3=res["tool"]["v3"]))
batch = pd.DataFrame(rows)
print(batch.to_string(index=False))
num = batch[[c for c in ("正交度前", "正交度后") if c in batch]]
print("\n★ 平均正交度：%.4f → %.4f ✓（平均提升 %+.4f ✓）"
      % (num["正交度前"].mean(), num["正交度后"].mean(),
         (num["正交度后"] - num["正交度前"]).mean()))
print("★ 平均最歪：%.4f mm → %.4f mm ✓"
      % (batch["最歪前_mm"].mean(), batch["最歪后_mm"].mean()))
print("\n结论：工具**能把该直的拉直** ✓；剩下没动的（如 `RC` 网标签那根 0.038 mm ✓）"
      "是**脚位未知**的安全阀拦下的 ✓ —— 工具自己会说清楚 ✓，人再看一眼就够 ✓。")
